----
# **<span style="color:DarkSlateBlue">PROYECTO BOOKING_REVIEWS</span>**
# **<span style="color:DarkSlateBlue">Limpieza de datos</span>**
----

---
---
## <span style="color:gray">**1. Importación de librerías**</span> 📂

In [281]:
# Tratamiento de datos
import numpy as np
import pandas as pd 
from IPython.display import display
import pycountry

# Visualización de datos
import seaborn as sns
import matplotlib.pyplot as plt

# Configuración de ruta
import sys
sys.path.append('../')

# Importación de funciones personalizadas
from src.soporte2_limpieza import *


---
---
## <span style="color:gray">**2. Carga de datos**</span> 📥

In [282]:
booking_raw = pd.read_csv("../data/raw/hotel_booking.csv")

reviews_raw = pd.read_csv("../data/raw/hotel_reviews.csv")

In [283]:
# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

---
---

----
# <span style="color:DarkSlateBlue">**Desarrollo del proyecto - 2**</span>
----


---
---
## <span style="color:gray">**Limpieza de datos**</span> 🧹

Antes de continuar con el análisis exploratorio de datos (EDA), creamos copias de los DataFrames originales.
Esto nos permite manipular, limpiar o transformar los datos sin afectar los originales, asegurando que siempre podamos recuperar la información original si es necesario.

In [284]:
booking_limpio = booking_raw.copy()
reviews_limpio = reviews_raw.copy()

---
## `booking_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>


La única columna que habría que estandarizar es `phone-number`, ya que utiliza como separador "-", en lugar de "_" como el resto de nombres de columnas. Sin embargo, al ser irrelevante para nuestro análisis, la eliminaremos en el próximo paso.

### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

- `arrival_date_week_number` y `arrival_date_day_of_month`

Solo necesitamos el mes y año de llegada para poder unir este dataset con el de reviews. Por lo tanto, estas columnas se pueden eliminar.

- `agent`y `company`

Representan los ID de las agencias y compañías. No aportan información relevante para nuestro análisis, por lo que se eliminan.

- `days_in_waiting_list`

La mayoría de los huéspedes tienen 0 días en espera y no es una información útil para el análisis que queremos realizar. En consecuncia, la eliminaremos.

- `required_car_parking_spaces` y `total_of_special_requests`

El valor promedio es muy bajo, la desviación estándar es baja y los percentiles muestran que la mayoría de los huéspedes no requiere estas opciones. Estas columnas aportan poca información útil y se eliminan.

- `name`, `email`, `phone-number`, `credit_card` 

Nos dan información sensible del huésped que no es que no aporta valor al análisis. Se eliminan.

**Eliminamos las columnas anteriores**

In [285]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
booking_limpio = eliminar_columns(
    booking_limpio,
    ["arrival_date_week_number", "arrival_date_day_of_month", 
     "agent", "company", "days_in_waiting_list", 
     "required_car_parking_spaces", "total_of_special_requests",
     "name", "email", "phone-number", "credit_card"]
)

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Cambiamos la abreviatura de los países por el nombre completo del país.**

In [286]:
def code_to_name(code):
    try:
        return pycountry.countries.get(alpha_3=code.upper()).name
    except:
        return None

booking_limpio['country'] = booking_limpio['country'].apply(code_to_name)

- **Comprobamos los datos únicos de las variables categóricas y hacemos la limpieza de algunos de los datos.**

In [287]:
col_cat = booking_limpio.select_dtypes(include=['category', 'str']).columns
datos_unicos(booking_limpio, col_cat)



Los datos únicos de la varible hotel son:

 <StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible arrival_date_month son:

 <StringArray>
[     'July',    'August', 'September',   'October',  'November',  'December',
   'January',  'February',     'March',     'April',       'May',      'June']
Length: 12, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible meal son:

 <StringArray>
['BB', 'FB', 'HB', 'SC', 'Undefined']
Length: 5, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible country son:

 <StringArray>
[                        'Portugal',                   'United Kingdom',
                    'United States',                            'Spain',
                          'Ireland',                           'France',
                             

- Todas las variables presentan mayúsculas y minúsculas, lo estandarizamos todo a minúsculas.
- Las variables `hotel`, `deposit_type` y `market_segment` tienen algunos datos con espacio entre medias, lo cambiamos por "_".
- `market_segment` y `distribution_channel` tienen el caracter "/", el cual sustituimos por "_".
- `customer_type` y `reservation_status` tienen como separador de palabras "-", lo cambiamos por "_".


In [288]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
minus(booking_limpio, col_cat)

# Llamamos a la función reemplazar guardada en el archivo de soporte 
reemplazar(booking_limpio, ["hotel", "country", "deposit_type", "market_segment"], " ", "_")
reemplazar(booking_limpio, ["market_segment", "distribution_channel"], "/", "_")
reemplazar(booking_limpio, ["customer_type", "reservation_status"], "-", "_")

### <span style="color:darkgray">**4. Corrección de tipos de datos**</span>

- **Cambiamos las variables `is_canceled` y `is_repeated_guest` a variables categóricas:** 

   - 1 = 'si'
   - 0 = 'no'

In [289]:
columnas = ["is_canceled", "is_repeated_guest"]

for columna in columnas:
    booking_limpio[columna] = booking_limpio[columna].replace({1:'si', 0:'no'})

- **Variable `children`**

Para convertir esta variable a números enteros, primero hay que pasar de string a float, hacer un redondeo al entero más cercano y finalmente se cambia a int:

In [290]:
booking_limpio['children'] = booking_limpio['children'].astype('Int64')

- **Variable `reservation_status_date`**

In [291]:
booking_limpio["reservation_status_date"] = (
    pd.to_datetime(booking_limpio["reservation_status_date"], 
                   format="%Y-%M-%d", 
                   errors="coerce")
                   )

Una vez convertida la variable a formato fecha, se crearán dos nuevas variables: una que contenga el año y otra el mes, con el objetivo de obtener información temporal más relevante para el análisis.

In [292]:
booking_limpio['reservation_status_year'] = booking_limpio['reservation_status_date'].dt.year

In [293]:
booking_limpio['reservation_status_month'] = booking_limpio['reservation_status_date'].dt.month_name()

month_year_arrival

### <span style="color:darkgray">**5. Tratamiento de valores nulos**</span>

Durante la exploración inicial del Dataset, identificamos columnas con valores nulos; a continuación, evaluaremos cómo tratarlos individualmente.

Antes de tratar los nulos generamos un DataFrame que contenga solo las columnas que tienen nulos:

In [294]:
df_nulos = booking_limpio.loc[:, booking_limpio.isnull().sum() > 0]
df_nulos.columns

Index(['children', 'country'], dtype='str')

Recordamos el porcentaje de nulos de cada columna:

In [295]:
(df_nulos.isnull().sum() / (df_nulos.shape[0])*100).round(3)

children    0.003
country     1.483
dtype: float64

- Variable `children`

Lo imputamos con la moda.

In [296]:
booking_limpio["children"] = booking_limpio["children"].fillna(booking_limpio["children"].mode()[0])


- Variable `country`

Sustituímos los valores nulos por 'unknown'.

In [297]:
booking_limpio["country"] = booking_limpio["country"].fillna('unknown') 

---
## `reviews_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>

- Estandarización de nombres de las columnas

In [298]:
estandar_columns(reviews_limpio)

### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

- `review_total_negative_word_counts` y `review_total_positive_word_counts` 
- `days_since_review`
- `lat` y `lng`

**Eliminamos las columnas anteriores**

In [299]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
reviews_limpio = eliminar_columns(
    reviews_limpio, 
    ["review_total_negative_word_counts", 
     "review_total_positive_word_counts",
     "days_since_review", 
     "lat", "lng"]
     )

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Cambiamos los datos de las variables categóricas a minúscula.**

In [300]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
col_cat = reviews_limpio.select_dtypes(include=['category', 'str']).columns
minus(reviews_limpio, col_cat)

- **Creamos dos nuevas columnas a partir de `hotel_address`:**

    - `country`
    - `city`

In [301]:
reviews_limpio['country'] = reviews_limpio['hotel_address'].apply(lambda x: x.split()[-1])
reviews_limpio['city'] = reviews_limpio['hotel_address'].apply(lambda x: x.split()[-2])

Eliminamos la columna `hotel_address`

In [302]:
reviews_limpio = eliminar_columns(reviews_limpio, ["hotel_address"])

- **Cambiamos los espacios de las columnas `hotel_name` y `reviewer_nationality` por "_".**


In [303]:
reemplazar(reviews_limpio, ["hotel_name", "reviewer_nationality"], " ", "_")

- **Variables `positive_reviews` y `negative_reviews`.**

In [304]:
# Función para clasificar texto en categorías
def clasificar_review(texto):
    
    if any(word in texto for word in ['clean', 'dirty']):
        return 'cleanliness'
    elif any(word in texto for word in ['staff', 'service', 'check in', 'check out']):
        return 'staff'
    elif any(word in texto for word in ['location', 'area', 'distance']):
        return 'location'
    elif any(word in texto for word in ['room', 'bed', 'bathroom', 'shower']):
        return 'room'
    elif any(word in texto for word in ['ac', 'air_condition']):
        return 'air_conditioning'
    elif any(word in texto for word in ['amenities', 'mattress', 'pillow']):
        return 'amenities'
    elif any(word in texto for word in ['breakfast', 'food', 'restaurant']):
        return 'food'
    elif any(word in texto for word in ['price', 'value', 'expensive', 'refund']):
        return 'value'
    elif any(word in texto for word in ['wifi', 'tv', 'pool', 'gym', 'parking']):
        return 'facilities'
    elif any(word in texto for word in ['noise', 'noisy' 'quiet']):
        return 'noise'
    elif any(word in texto for word in ['construction', 'renovation']):
        return 'construction'
    elif any(word in texto for word in ['nothing', 'anything', 'negtive', 'positive']):
        return 'no_comment'
    else:
        return 'other'


# Aplicar a columnas de reviews
reviews_limpio['positive_tag'] = reviews_limpio['positive_review'].apply(clasificar_review)
reviews_limpio['negative_tag'] = reviews_limpio['negative_review'].apply(clasificar_review)

- **Variable `tags`.**

Creamos nuevas columnas con la información que contiene la columna.

In [305]:
# Cambiamos los string a lista de strings
import ast

reviews_limpio['tags'] = reviews_limpio['tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [306]:
def procesar_tags(lista): 
    
    result = { 'trip_type': None, 
              'group_type': None, 
              'room_type': None, 
              'nights': None, 
              'device': None } 
    
    for item in lista: 
        if ('leisure trip' in item) or ('business trip' in item):
            result['trip_type'] = item
        elif 'night' in item:
            result['nights'] = item
        elif 'submitted' in item:
            result['device'] = item
        elif ('couple' in item) or ('family with' in item) or ('traveler' in item) or ('group' in item) or ('with friends' in item):
            result['group_type'] = item
        else:
            result['room_type'] = item
    return pd.Series(result)

In [307]:
reviews_limpio = reviews_limpio.join(reviews_limpio['tags'].apply(procesar_tags))

In [308]:
reviews_limpio = eliminar_columns(reviews_limpio, ["tags"])

### <span style="color:darkgray">**4. Corrección de tipos de datos**</span>

- **Crear la columna `month_year_review` a partir de review date.**
Esto nos permitirá unir el dataset con el de bookings.

In [309]:
reviews_limpio['review_date'] = pd.to_datetime(reviews_limpio['review_date'])

In [310]:
reviews_limpio['review_year'] = reviews_limpio['review_date'].dt.year

In [311]:
reviews_limpio['review_month'] = reviews_limpio['review_date'].dt.month_name()

### <span style="color:darkgray">**5. Tratamiento de valores nulos**</span>

In [312]:
df_nulos = reviews_limpio.loc[:, reviews_limpio.isnull().sum() > 0]
df_nulos.columns

Index(['trip_type', 'room_type', 'nights', 'device'], dtype='str')

In [313]:
(df_nulos.isnull().sum() / (df_nulos.shape[0])*100).round(3)

trip_type     2.913
room_type     0.041
nights        0.037
device       40.350
dtype: float64

- `room_type` lo eliminamos porque hay muchos datos únicos y no es información relevante para nuestro análisis

- `device` el único dato que tenemos es *mobile*, así que como no disponemos información del resto de devices, la eliminamos

In [314]:
reviews_limpio = eliminar_columns(reviews_limpio, ["room_type", "device"])

- trip_type lo rellenamos con *unknown*

In [315]:
reviews_limpio["trip_type"] = reviews_limpio["trip_type"].fillna('unknown')

- nights lo rellenamos con la moda

In [316]:
reviews_limpio["nights"] = reviews_limpio["nights"].fillna(reviews_limpio["nights"].mode()[0])

### <span style="color:darkgray">**6. Eliminación de duplicados**</span>

In [317]:
reviews_limpio = reviews_limpio.drop_duplicates()

---
---
## <span style="color:gray">**Validación final de los datasets**</span> ✅

- **Comprobamos que la limpieza de ambos dataset se ha realizado correctamente**


### **`booking_limpio`** 

In [318]:
# Llamamos a la función analisis_rapido guardada en el archivo de soporte
analisis_rapido(booking_limpio)



Las 3 primeras filas del Dataframe son:



,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,customer_type,adr,reservation_status,reservation_status_date,reservation_status_year,reservation_status_month
0,resort_hotel,no,342,2015,july,0,0,2,0,0,bb,portugal,direct,direct,no,0,0,c,c,3,no_deposit,transient,0.0,check_out,2015-01-01 00:07:00,2015,January
1,resort_hotel,no,737,2015,july,0,0,2,0,0,bb,portugal,direct,direct,no,0,0,c,c,4,no_deposit,transient,0.0,check_out,2015-01-01 00:07:00,2015,January
2,resort_hotel,no,7,2015,july,0,1,1,0,0,bb,united_kingdom,direct,direct,no,0,0,a,c,0,no_deposit,transient,75.0,check_out,2015-01-02 00:07:00,2015,January



-----------------------------------------------------------

La información básica del Dataframe es la siguiente:

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 27 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  str           
 1   is_canceled                     119390 non-null  object        
 2   lead_time                       119390 non-null  int64         
 3   arrival_date_year               119390 non-null  int64         
 4   arrival_date_month              119390 non-null  str           
 5   stays_in_weekend_nights         119390 non-null  int64         
 6   stays_in_week_nights            119390 non-null  int64         
 7   adults                          119390 non-null  int64         
 8   children                        119390 non-null  Int64         
 9   babies              

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
customer_type                     0
adr                               0
reservation_status                0
reservation_status_date           0
reservation_status_year           0
reservation_status_month          0
dtype: int64

Y los datos de la variable `reservation_status_month` empiezan por mayúsculas, los cambiamos a minúsculas para mantener la consistencia en todo el DataFrame:

In [319]:
minus(booking_limpio, ['reservation_status_month'])

### **`reviews_limpio`** 

In [323]:
# Llamamos a la función analisis_rapido guardada en el archivo de soporte
analisis_rapido(reviews_limpio)



Las 3 primeras filas del Dataframe son:



,additional_number_of_scoring,review_date,average_score,hotel_name,reviewer_nationality,negative_review,total_number_of_reviews,positive_review,total_number_of_reviews_reviewer_has_given,reviewer_score,country,city,positive_tag,negative_tag,trip_type,group_type,nights,review_year,review_month
0,194,2017-08-03,7.7,hotel_arena,russia,i am so angry that i made this post available via all possible sites i use when planing my trips so no one will make the mistake of booking this place i made my booking via booking com we stayed for 6 nights in this hotel from 11 to 17 july upon arrival we were placed in a small room on the 2nd floor of the hotel it turned out that this was not the room we booked i had specially reserved the 2 level duplex room so that we would have a big windows and high ceilings the room itself was ok if you don t mind the broken window that can not be closed hello rain and a mini fridge that contained some sort of a bio weapon at least i guessed so by the smell of it i intimately asked to change the room and after explaining 2 times that i booked a duplex btw it costs the same as a simple double but got way more volume due to the high ceiling was offered a room but only the next day so i had to check out the next day before 11 o clock in order to get the room i waned to not the best way to begin your holiday so we had to wait till 13 00 in order to check in my new room what a wonderful waist of my time the room 023 i got was just as i wanted to peaceful internal garden view big window we were tired from waiting the room so we placed our belongings and rushed to the city in the evening it turned out that there was a constant noise in the room i guess it was made by vibrating vent tubes or something it was constant and annoying as hell and it did not stop even at 2 am making it hard to fall asleep for me and my wife i have an audio recording that i can not attach here but if you want i can send it via e mail the next day the technician came but was not able to determine the cause of the disturbing sound so i was offered to change the room once again the hotel was fully booked and they had only 1 room left the one that was smaller but seems newer,1403,only the park outside of the hotel was beautiful,7,2.9,netherlands,amsterdam,other,staff,leisure trip,couple,stayed 6 nights,2017,August
1,194,2017-08-03,7.7,hotel_arena,ireland,no negative,1403,no real complaints the hotel was great great location surroundings rooms amenities and service two recommendations however firstly the staff upon check in are very confusing regarding deposit payments and the staff offer you upon checkout to refund your original payment and you can make a new one bit confusing secondly the on site restaurant is a bit lacking very well thought out and excellent quality food for anyone of a vegetarian or vegan background but even a wrap or toasted sandwich option would be great aside from those minor minor things fantastic spot and will be back when i return to amsterdam,7,7.5,netherlands,amsterdam,staff,other,leisure trip,couple,stayed 4 nights,2017,August
2,194,2017-07-31,7.7,hotel_arena,australia,rooms are nice but for elderly a bit difficult as most rooms are two story with narrow steps so ask for single level inside the rooms are very very basic just tea coffee and boiler and no bar empty fridge,1403,location was good and staff were ok it is cute hotel the breakfast range is nice will go back,9,7.1,netherlands,amsterdam,staff,room,leisure trip,family with young children,stayed 3 nights,2017,July



-----------------------------------------------------------

La información básica del Dataframe es la siguiente:

<class 'pandas.DataFrame'>
Index: 515201 entries, 0 to 515737
Data columns (total 19 columns):
 #   Column                                      Non-Null Count   Dtype         
---  ------                                      --------------   -----         
 0   additional_number_of_scoring                515201 non-null  int64         
 1   review_date                                 515201 non-null  datetime64[us]
 2   average_score                               515201 non-null  float64       
 3   hotel_name                                  515201 non-null  str           
 4   reviewer_nationality                        515201 non-null  str           
 5   negative_review                             515201 non-null  str           
 6   total_number_of_reviews                     515201 non-null  int64         
 7   positive_review                             515201 non-

additional_number_of_scoring                  0
review_date                                   0
average_score                                 0
hotel_name                                    0
reviewer_nationality                          0
negative_review                               0
total_number_of_reviews                       0
positive_review                               0
total_number_of_reviews_reviewer_has_given    0
reviewer_score                                0
country                                       0
city                                          0
positive_tag                                  0
negative_tag                                  0
trip_type                                     0
group_type                                    0
nights                                        0
review_year                                   0
review_month                                  0
dtype: int64

Hemos verificado que la limpieza de ambos dataframes se ha realizado correctamente: 

- Los nombres de las columnas son claros y descriptivos.
- Los datos son consistentes entre sí.
- No se observan valores nulos.
- Las variables presentan formatos adecuados para análisis y modelado posteriores .

Además, las transformaciones aplicadas (como estandarización de texto, conversión de tipos numéricos y codificación de variables categóricas) aseguran que ambos dataframes puedan combinarse y utilizarse de manera confiable.

- **Guardamos los dataset limpios**

In [321]:
#booking_limpio.to_csv("../data/output/booking_limpio.csv", index=False)
#reviews_limpio.to_csv("../data/output/reviews_limpio.csv", index=False)

---